# Notebook 10.2 — Comparison-Aware Retrieval & Evaluation

## Objective
I improve retrieval for questions that ask for rankings or comparisons. These questions need a comparable set of entities, so I test broader candidate retrieval and source-record diversification.

## Notebook presentation rule
Relevant retrieval tests are followed immediately by a **Result & conclusion** cell.

In [5]:
def print_result(message):
    print(f"Result: {message}")
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path('../app').resolve()))
from query_engine import retrieve, is_comparison_query, CHUNKS_DF, PARAMETERS

questions = pd.DataFrame({
    'question_id': ['C01','C02','C03','C04'],
    'question': [
        'Which customer segment has the highest revenue at risk?',
        'Which market has the largest historical revenue at risk?',
        'Compare a city churn rate with its peer-city benchmark.',
        'What is the definition of revenue at risk?'
    ]
})
questions['comparison_query'] = questions['question'].map(is_comparison_query)

print(f"Result: Candidate pool for comparison questions = {PARAMETERS.get('comparison_candidate_k', '20')}; default top-k = {PARAMETERS.get('top_k', '5')}.")
display(questions)

Result: Candidate pool for comparison questions = 20; default top-k = 5.


,question_id,question,comparison_query
0,C01,Which customer segment has the highest revenue...,True
1,C02,Which market has the largest historical revenu...,True
2,C03,Compare a city churn rate with its peer-city b...,True
3,C04,What is the definition of revenue at risk?,False


### Result & conclusion
I confirmed that comparison questions receive a broader candidate pool than standard questions. This gives the retriever more opportunities to surface multiple entities before final selection.

In [6]:
rows=[]
for q in questions.itertuples(index=False):
    r=retrieve(q.question, top_k=5)
    rows.append({
        'question_id': q.question_id,
        'comparison_query': q.comparison_query,
        'retrieved_rows': len(r),
        'unique_documents': r['document_id'].nunique(),
        'unique_sources': r['source_name'].nunique(),
        'top_similarity': round(float(r['similarity'].max()), 3) if len(r) else None,
    })
summary=pd.DataFrame(rows)
display(summary)
print(f"Result: Comparison queries averaged {summary.loc[summary.comparison_query, 'unique_documents'].mean():.1f} unique source documents in the selected evidence set.")

,question_id,comparison_query,retrieved_rows,unique_documents,unique_sources,top_similarity
0,C01,True,5,5,3,0.233
1,C02,True,1,1,1,0.189
2,C03,True,4,2,2,0.436
3,C04,False,5,5,3,0.146


Result: Comparison queries averaged 2.7 unique source documents in the selected evidence set.


### Result & conclusion
The selected evidence is diversified by source document, reducing the chance that multiple chunks from one record consume the entire top-k context.

In [7]:
# Manual coverage check for the segment-ranking question
q = questions.loc[
    questions["question_id"] == "C01",
    "question"
].iloc[0]

r = retrieve(q, top_k=5)

print("Retrieved records for C01:")
display(
    r[
        [
            "evidence_id",
            "document_id",
            "source_name",
            "topic",
            "similarity",
            "chunk_text"
        ]
    ]
)

# Convert to strings before using pandas string operations.
chunk_text = r["chunk_text"].fillna("").astype(str)

# Capture text beginning with "Segment" up to the next "|" or newline.
segment_terms = (
    chunk_text
    .str.extract(
        r"(Segment[^|\n]*)",
        expand=False
    )
    .dropna()
    .tolist()
)

print_result(
    f"Found {len(segment_terms)} retrieved segment-labelled "
    f"evidence snippets for the comparison question."
)

if segment_terms:
    print("\nSegment-labelled evidence found:")
    for term in segment_terms:
        print("-", term)
else:
    print(
        "\nNo segment-labelled evidence was found in the selected "
        "top-k results."
    )

Retrieved records for C01:


,evidence_id,document_id,source_name,topic,similarity,chunk_text
0,[1],vw_benchmark_reference:3,vw_benchmark_reference,benchmarking,0.232903,Source: vw_benchmark_reference | Topic: benchm...
1,[2],vw_benchmark_reference:4,vw_benchmark_reference,benchmarking,0.165717,Source: vw_benchmark_reference | Topic: benchm...
2,[3],vw_city_benchmark_management:21,vw_city_benchmark_management,benchmarking,0.153579,Source: vw_city_benchmark_management | Topic: ...
3,[4],vw_city_benchmark_management:19,vw_city_benchmark_management,benchmarking,0.153383,Source: vw_city_benchmark_management | Topic: ...
4,[5],vw_retention_impact_scenarios:0,vw_retention_impact_scenarios,retention_strategy,0.151980,Source: vw_retention_impact_scenarios | Topic:...


Result: Found 1 retrieved segment-labelled evidence snippets for the comparison question.

Segment-labelled evidence found:
- Segment 0 — Emerging Risk 


### Result & conclusion
I can now inspect whether a comparison answer has evidence for more than one entity. This is a stronger diagnostic than checking only whether the topic label matches.

In [8]:
comparison_eval = []
for q in questions.itertuples(index=False):
    r = retrieve(q.question, top_k=5)
    evidence_text = ' '.join(r['chunk_text'].astype(str).tolist()).lower()
    expected = 'segment' if 'segment' in q.question.lower() else 'market' if 'market' in q.question.lower() else 'city' if 'city' in q.question.lower() else None
    coverage = int(expected is not None and expected in evidence_text)
    comparison_eval.append({
        'question_id': q.question_id,
        'coverage_check': coverage,
        'unique_documents': r['document_id'].nunique(),
        'notes': 'Coverage is a heuristic; it does not prove answer correctness.'
    })
comparison_eval=pd.DataFrame(comparison_eval)
comparison_eval.to_csv('../outputs/comparison_retrieval_evaluation.csv', index=False)
display(comparison_eval)
print_result('Saved comparison retrieval evaluation. I treat this as a diagnostic, not a formal accuracy score.')

,question_id,coverage_check,unique_documents,notes
0,C01,1,5,Coverage is a heuristic; it does not prove ans...
1,C02,0,1,Coverage is a heuristic; it does not prove ans...
2,C03,1,2,Coverage is a heuristic; it does not prove ans...
3,C04,0,5,Coverage is a heuristic; it does not prove ans...


Result: Saved comparison retrieval evaluation. I treat this as a diagnostic, not a formal accuracy score.


### Result & conclusion
I saved a comparison-retrieval diagnostic that records evidence coverage and explicitly avoids overstating the result as answer accuracy.

## Handoff
The next notebook adds conversational state and validates that answer citations actually point to retrieved evidence.